In [ ]:
# ============================================
# 1. LOAD AND INSPECT DATA
# ============================================

# Load the video game sales dataset and preview the first few rows
import pandas as pd

df = pd.read_csv("vgsales.csv")
df.head()


In [ ]:
# Check the shape of the dataset (number of rows and columns)
df.shape


In [ ]:
# Inspect column types and non-null counts
df.info()


In [ ]:
# Count missing values per column
df.isnull().sum()


In [ ]:
# Get basic statistical summary for numerical columns
df.describe()



In [ ]:
# ============================================
# 2. DATA CLEANING (MISSING VALUES)
# ============================================

# Fill missing Year values using the median Year per Platform
df['Year'] = df.groupby('Platform')['Year'].transform(
    lambda x: x.fillna(x.median())
)


In [ ]:
# Fill missing Publisher values with 'Unknown'
df['Publisher'] = df['Publisher'].fillna("Unknown")


In [ ]:
# Convert Year to integer type after imputation
df['Year'] = df['Year'].astype(int)


In [ ]:
# Verify there are no remaining missing values
df.isnull().sum()



In [ ]:
# ============================================
# 3. EXPLORATORY DATA ANALYSIS (EDA)
# ============================================

# Plot histograms for regional and global sales with labeled axes
import matplotlib.pyplot as plt

numeric_cols = ["NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales", "Global_Sales"]

df[numeric_cols].hist(bins=30, figsize=(12, 8))

for ax in plt.gcf().axes:
    ax.set_xlabel("Sales (in millions)")
    ax.set_ylabel("Number of Games")

plt.tight_layout()
plt.show()


In [ ]:
# Plot total global sales by Genre as a bar chart
genre_sales = df.groupby("Genre")["Global_Sales"].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
genre_sales.plot(kind="bar")
plt.xlabel("Genre")
plt.ylabel("Total Global Sales (in millions)")
plt.title("Total Global Sales by Genre")
plt.tight_layout()
plt.show()


In [ ]:
# Plot total global sales by Platform as a bar chart
platform_sales = df.groupby("Platform")["Global_Sales"].sum().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
platform_sales.plot(kind="bar")
plt.xlabel("Platform")
plt.ylabel("Total Global Sales (in millions)")
plt.title("Total Global Sales by Platform")
plt.tight_layout()
plt.show()


In [ ]:
# Plot the time trend of total global sales over the years
year_sales = df.groupby("Year")["Global_Sales"].sum()

plt.figure(figsize=(12, 6))
plt.plot(year_sales.index, year_sales.values, marker="o")
plt.xlabel("Year")
plt.ylabel("Total Global Sales (in millions)")
plt.title("Global Video Game Sales Over Time")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Compute and visualize the correlation matrix of sales features
corr = df[["NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales", "Global_Sales"]].corr()

plt.figure(figsize=(8, 6))
plt.imshow(corr, interpolation="nearest")
plt.colorbar()

plt.xticks(range(len(corr.columns)), corr.columns, rotation=45)
plt.yticks(range(len(corr.columns)), corr.columns)

for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        plt.text(j, i, f"{corr.iloc[i, j]:.2f}",
                 ha="center", va="center", color="black")

plt.title("Correlation Matrix of Sales Features")
plt.tight_layout()
plt.show()



In [ ]:
# ============================================
# 4. INITIAL (LEAKY) MODEL SETUP — FOR DEMO ONLY
# ============================================

# Define target and features including regional sales (this causes leakage!)
y = df["Global_Sales"]
X = df.drop(columns=["Global_Sales", "Name", "Rank"])

print("Feature matrix shape (X):", X.shape)
print("Target vector shape (y):", y.shape)
X.head()


In [ ]:
# One-hot encode categorical features for the leaky setup (demonstration only)
X_encoded = pd.get_dummies(X, columns=["Platform", "Genre", "Publisher"])

print("Encoded feature matrix shape:", X_encoded.shape)
X_encoded.head()


In [ ]:
# Train/test split for the leaky setup (demonstration only)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


In [ ]:
# Train and evaluate a Linear Regression model using leaky features
# This is kept only to show why leakage leads to unrealistically high R².
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

y_pred_lr = lin_reg.predict(X_test)

mae_lr = mean_absolute_error(y_test, y_pred_lr)
mse_lr = mean_squared_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mse_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print("Linear Regression Performance (WITH LEAKAGE):")
print("MAE :", mae_lr)
print("MSE :", mse_lr)
print("RMSE:", rmse_lr)
print("R²  :", r2_lr)



In [ ]:
# ============================================
# 5. CLEAN MODELING PIPELINE (NO LEAKAGE)
# ============================================

# Rebuild feature matrix using only pre-release metadata (no regional sales)
X = df[["Platform", "Year", "Genre", "Publisher"]]
y = df["Global_Sales"]

print("X shape:", X.shape)
print("y shape:", y.shape)
X.head()


In [ ]:
# One-hot encode the cleaned feature set (only metadata-based features)
X_encoded = pd.get_dummies(X, columns=["Platform", "Genre", "Publisher"])

print("Encoded feature matrix shape:", X_encoded.shape)
X_encoded.head()


In [ ]:
# Train/test split for the clean, non-leaky setup
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)


In [ ]:
# Train and evaluate Linear Regression on the clean, non-leaky features
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

y_pred_lr = lin_reg.predict(X_test)

mae_lr = mean_absolute_error(y_test, y_pred_lr)
mse_lr = mean_squared_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mse_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print("Linear Regression Performance (NO LEAKAGE):")
print("MAE :", mae_lr)
print("MSE :", mse_lr)
print("RMSE:", rmse_lr)
print("R²  :", r2_lr)


In [ ]:
# Train and evaluate a Decision Tree Regressor on the clean features
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

mae_dt = mean_absolute_error(y_test, y_pred_dt)
mse_dt = mean_squared_error(y_test, y_pred_dt)
rmse_dt = np.sqrt(mse_dt)
r2_dt = r2_score(y_test, y_pred_dt)

print("Decision Tree Performance (NO LEAKAGE):")
print("MAE :", mae_dt)
print("MSE :", mse_dt)
print("RMSE:", rmse_dt)
print("R²  :", r2_dt)


In [ ]:
# Train and evaluate a Random Forest Regressor on the clean features
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mse_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest Performance (NO LEAKAGE):")
print("MAE :", mae_rf)
print("MSE :", mse_rf)
print("RMSE:", rmse_rf)
print("R²  :", r2_rf)



In [ ]:
# ============================================
# 6. MODEL COMPARISON TABLE
# ============================================

# Create a comparison table summarizing performance of all clean models
model_results = [
    ["Linear Regression", mae_lr, rmse_lr, r2_lr],
    ["Decision Tree", mae_dt, rmse_dt, r2_dt],
    ["Random Forest", mae_rf, rmse_rf, r2_rf],
]

results_df = pd.DataFrame(model_results, columns=["Model", "MAE", "RMSE", "R²"])
print("Model comparison on clean, non-leaky setup:")
print(results_df)



In [ ]:
# ============================================
# 7. FEATURE IMPORTANCE (RANDOM FOREST)
# ============================================

# Compute and sort feature importances from the trained Random Forest model
importances = rf.feature_importances_
feature_names = X_encoded.columns

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

importance_df.head(20)


In [ ]:
# Visualize the top 20 most important features as a horizontal bar chart
top_n = 20
top_features = importance_df.head(top_n)

plt.figure(figsize=(10, 8))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.title(f"Top {top_n} Most Important Features (Random Forest)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()



In [ ]:
# ============================================
# 8. PREDICTION QUALITY VISUALIZATION
# ============================================

# Plot predicted vs actual Global_Sales for the Random Forest model
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_rf, alpha=0.4)
plt.xlabel("Actual Global Sales (millions)")
plt.ylabel("Predicted Global Sales (millions)")
plt.title("Random Forest: Actual vs Predicted Global Sales")

min_val = min(y_test.min(), y_pred_rf.min())
max_val = max(y_test.max(), y_pred_rf.max())
plt.plot([min_val, max_val], [min_val, max_val], color="red", linestyle="--")

plt.tight_layout()
plt.show()
